## Install SQLAlchemy

This cell installs the SQLAlchemy library, which is an SQL toolkit and Object-Relational Mapper (ORM) for Python.

In [2]:
!pip install sqlalchemy

## Define ORM Models

This cell defines the Python classes (`Customer`, `Order`, `OrderItem`) that map to the database tables. It uses SQLAlchemy's DeclarativeBase to define the schema and relationships.

In [3]:
from typing import List, Optional
from sqlalchemy import ForeignKey, String, Numeric, create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship


class Base(DeclarativeBase):
    pass


class Customer(Base):
    __tablename__ = "customer"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(150), unique=True)

    orders: Mapped[List["Order"]] = relationship(
        back_populates="customer", cascade="all, delete-orphan"
    )

    def __repr__(self) -> str:
        return f"Customer(id={self.id}, name={self.name!r})"


class Order(Base):
    __tablename__ = "order"

    id: Mapped[int] = mapped_column(primary_key=True)
    customer_id: Mapped[int] = mapped_column(ForeignKey("customer.id"))
    status: Mapped[str] = mapped_column(String(20), default="pending")

    customer: Mapped["Customer"] = relationship(back_populates="orders")
    items: Mapped[List["OrderItem"]] = relationship(
        back_populates="order", cascade="all, delete-orphan"
    )

    def __repr__(self) -> str:
        return f"Order(id={self.id}, status={self.status!r})"


class OrderItem(Base):
    __tablename__ = "order_item"

    id: Mapped[int] = mapped_column(primary_key=True)
    order_id: Mapped[int] = mapped_column(ForeignKey("order.id"))
    product_name: Mapped[str] = mapped_column(String(200))
    quantity: Mapped[int]
    unit_price: Mapped[float] = mapped_column(Numeric(10, 2))

    order: Mapped["Order"] = relationship(back_populates="items")

    def __repr__(self) -> str:
        return f"OrderItem(product={self.product_name!r}, qty={self.quantity})"

## Initialize Database Engine and Create Tables

This cell creates an SQLite database named `orders.db` and initializes the SQLAlchemy engine. It then uses `Base.metadata.create_all(engine)` to create all defined tables in the database.

In [4]:
engine = create_engine("sqlite:///orders.db", echo=False)
Base.metadata.create_all(engine)

## Basic Session Usage

This cell demonstrates how to create and use a SQLAlchemy session to interact with the database. A session manages the persistence operations for objects.

In [5]:
from sqlalchemy.orm import Session

with Session(engine) as session:
    # all work happens here
    session.commit()

## Add Initial Data

This cell adds sample `Customer` and `Order` data, including associated `OrderItem` details, to the database. It then commits these changes to persist them.

In [6]:
with Session(engine) as session:
    rahul = Customer(
        name="Rahul Sharma",
        email="rahul.sharma@example.com"
    )
    priya = Customer(
        name="Priya Nair",
        email="priya.nair@example.com"
    )

    order = Order(
        customer=rahul,
        status="confirmed",
        items=[
            OrderItem(product_name="Mechanical Keyboard", quantity=1, unit_price=129.99),
            OrderItem(product_name="USB-C Hub", quantity=2, unit_price=34.50),
        ]
    )

    session.add_all([rahul, priya, order])
    session.commit()

## Query All Customers

This cell demonstrates how to query and retrieve all `Customer` records from the database using the `select` construct and `session.scalars().all()`.

In [7]:
from sqlalchemy import select

with Session(engine) as session:
    customers = session.scalars(select(Customer)).all()
    for c in customers:
        print(c)

Customer(id=1, name='Rahul Sharma')
Customer(id=2, name='Priya Nair')


## Query Confirmed Orders

This cell queries for `Order` records where the `status` is 'confirmed'. It uses a `where` clause to filter the results.

In [8]:
with Session(engine) as session:
    confirmed = session.scalars(
        select(Order).where(Order.status == "confirmed")
    ).all()
    print(confirmed)

[Order(id=1, status='confirmed')]


## Query Orders by Customer Email with Joins

This cell shows how to perform a query that involves joining `Order` and `Customer` tables. It retrieves orders associated with a specific customer email and also loads their related `OrderItem`s.

In [9]:
with Session(engine) as session:
    stmt = (
        select(Order)
        .join(Order.customer)
        .where(Customer.email == "rahul.sharma@example.com")
    )
    orders = session.scalars(stmt).all()
    for order in orders:
        print(order, "→", order.items)

Order(id=1, status='confirmed') → [OrderItem(product='Mechanical Keyboard', qty=1), OrderItem(product='USB-C Hub', qty=2)]


## Update an Order's Status

This cell demonstrates how to fetch an existing `Order` record, modify its `status` attribute, and then commit the changes back to the database.

In [10]:
with Session(engine) as session:
    order = session.scalars(
        select(Order).where(Order.id == 1)
    ).one()

    order.status = "shipped"
    session.commit()
    print(order)

Order(id=1, status='shipped')


## Delete a Customer Record

This cell illustrates how to retrieve a `Customer` record and then delete it from the database. Due to `cascade='all, delete-orphan'` in the relationship, associated orders will also be deleted.

In [11]:
with Session(engine) as session:
    priya = session.scalars(
        select(Customer).where(Customer.email == "priya.nair@example.com")
    ).one()

    session.delete(priya)
    session.commit()
    print("Deleted:", priya.name)

Deleted: Priya Nair


## Calculate Total for Each Order with Eager Loading

This cell queries all `Order` records and uses `selectinload(Order.items)` to eagerly load the associated `OrderItem`s. It then calculates and prints the total cost for each order.

In [12]:
from sqlalchemy.orm import selectinload

with Session(engine) as session:
    orders = session.scalars(
        select(Order).options(selectinload(Order.items))
    ).all()

    for order in orders:
        total = sum(item.quantity * item.unit_price for item in order.items)
        print(f"Order {order.id}: ${total:.2f}")

Order 1: $198.99
